# Paclitaxel Multi-omics Classification

This notebook evaluates fold-safe supervised classifiers for paclitaxel response prediction using the cleaned RNA, CNV, and mutation matrices from `1-data_preprocessing.ipynb`.

Primary endpoint:

- **Resistant**: `PD` + `SD`
- **Sensitive**: `PR` + `CR`

Feature selection is performed inside each cross-validation fold. External gene priors, including the curated paclitaxel gene list and the NEST-VNN gene list, are fixed a priori and force-retained when present.

## Imports and Configuration

In [11]:
from pathlib import Path
import sys
import warnings

import numpy as np
import pandas as pd
from scipy.stats import mannwhitneyu, pointbiserialr

from sklearn.base import clone
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import GridSearchCV, RepeatedStratifiedKFold, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

warnings.filterwarnings("ignore")

# Works whether the notebook is launched from the repo root or from 4_paclitaxel/.
CWD = Path.cwd().resolve()
if (CWD / "feature_sets.py").exists():
    NOTEBOOK_DIR = CWD
    PROJECT_ROOT = CWD.parent
elif (CWD / "4_paclitaxel" / "feature_sets.py").exists():
    PROJECT_ROOT = CWD
    NOTEBOOK_DIR = CWD / "4_paclitaxel"
else:
    raise FileNotFoundError("Could not locate 4_paclitaxel/feature_sets.py")

DATA_DIR = PROJECT_ROOT / "data/procdata/4_paclitaxel"
RESULTS_DIR = PROJECT_ROOT / "results/4_paclitaxel"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

sys.path.append(str(NOTEBOOK_DIR))
from feature_sets import (
    add_binary_response,
    load_pax_genes,
    load_nest_genes,
    split_x_class_y,
    select_features_by_omics,
)

RANDOM_STATE = 42
N_SPLITS = 5
N_REPEATS = 5
POS_LABEL = 1  # Sensitive

print("Project root:", PROJECT_ROOT)
print("Data dir:", DATA_DIR)
print("Results dir:", RESULTS_DIR)


Project root: /Users/guanqiaofeng/Documents/BHK/XevaDB/Manuscript_gitrepo/XevaDB_case_study
Data dir: /Users/guanqiaofeng/Documents/BHK/XevaDB/Manuscript_gitrepo/XevaDB_case_study/data/procdata/4_paclitaxel
Results dir: /Users/guanqiaofeng/Documents/BHK/XevaDB/Manuscript_gitrepo/XevaDB_case_study/results/4_paclitaxel


## Load Data and Define Binary Response

In [12]:
rna_df = add_binary_response(pd.read_csv(DATA_DIR / "rna_logtpm.csv", index_col=0))
cnv_df = add_binary_response(pd.read_csv(DATA_DIR / "cnv.csv", index_col=0))
mutation_df = add_binary_response(pd.read_csv(DATA_DIR / "mutation.csv", index_col=0))
metadata_df = add_binary_response(pd.read_csv(DATA_DIR / "metadata.csv", index_col=0))

pax_genes = load_pax_genes()
nest_genes = load_nest_genes()

all3_ids = sorted(set(rna_df.index) & set(cnv_df.index) & set(mutation_df.index))
rna_paired_df = rna_df.loc[all3_ids]
cnv_paired_df = cnv_df.loc[all3_ids]
mutation_paired_df = mutation_df.loc[all3_ids]

for name, df in [
    ("metadata", metadata_df),
    ("rna_all", rna_df),
    ("rna_paired", rna_paired_df),
    ("cnv", cnv_paired_df),
    ("mutation", mutation_paired_df),
]:
    print(f"\n{name} {df.shape}")
    print(df["mRECIST"].value_counts().to_string())
    print(df["response_binary"].value_counts().to_string())

print(f"\nPaclitaxel prior genes: {len(pax_genes)}")
print("NEST prior genes:", len(nest_genes))



metadata (80, 5)
mRECIST
SD    26
CR    19
PR    18
PD    17
response_binary
Resistant    43
Sensitive    37

rna_all (67, 49353)
mRECIST
SD    21
CR    17
PR    15
PD    14
response_binary
Resistant    35
Sensitive    32

rna_paired (62, 49353)
mRECIST
SD    20
CR    16
PD    13
PR    13
response_binary
Resistant    33
Sensitive    29

cnv (62, 9961)
mRECIST
SD    20
CR    16
PD    13
PR    13
response_binary
Resistant    33
Sensitive    29

mutation (62, 13647)
mRECIST
SD    20
CR    16
PD    13
PR    13
response_binary
Resistant    33
Sensitive    29

Paclitaxel prior genes: 69
NEST prior genes: 718


## Response QC

This checks how the binary response aligns with continuous `angle` and `TGI` metrics.

In [13]:
qc_rows = []
for metric in ["angle", "TGI"]:
    resistant = metadata_df.loc[metadata_df["response_binary"] == "Resistant", metric]
    sensitive = metadata_df.loc[metadata_df["response_binary"] == "Sensitive", metric]
    u_stat, p_val = mannwhitneyu(resistant, sensitive, alternative="two-sided")
    corr, corr_p = pointbiserialr(metadata_df["response_binary_num"], metadata_df[metric])
    auc = roc_auc_score(metadata_df["response_binary_num"], metadata_df[metric])
    auprc = average_precision_score(metadata_df["response_binary_num"], metadata_df[metric])
    qc_rows.append({
        "metric": metric,
        "resistant_median": resistant.median(),
        "sensitive_median": sensitive.median(),
        "mann_whitney_p": p_val,
        "point_biserial_r": corr,
        "point_biserial_p": corr_p,
        "auroc": auc,
        "auprc": auprc,
    })

response_qc_df = pd.DataFrame(qc_rows)
response_qc_df.to_csv(RESULTS_DIR / "response_metric_qc.csv", index=False)
response_qc_df


,metric,resistant_median,sensitive_median,mann_whitney_p,point_biserial_r,point_biserial_p,auroc,auprc
0,angle,7.893697,155.669929,1.998069e-12,0.847979,3.306680e-23,0.958517,0.968855
1,TGI,0.479326,0.865891,6.280624e-09,0.556356,8.401679e-08,0.878693,0.896106


## Classifier and Metric Helpers

In [14]:
def classification_metrics(y_true, y_pred, y_score):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    specificity = tn / (tn + fp) if (tn + fp) > 0 else np.nan
    return {
        "auroc": roc_auc_score(y_true, y_score),
        "auprc": average_precision_score(y_true, y_score),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "accuracy": accuracy_score(y_true, y_pred),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "sensitivity": recall_score(y_true, y_pred, zero_division=0),
        "specificity": specificity,
        "tn": tn,
        "fp": fp,
        "fn": fn,
        "tp": tp,
    }


def positive_class_score(model, X):
    if hasattr(model, "predict_proba"):
        return model.predict_proba(X)[:, 1]
    if hasattr(model, "decision_function"):
        return model.decision_function(X)
    return model.predict(X)


def make_classification_models(random_state=RANDOM_STATE):
    return {
        "logistic_l2": (
            Pipeline([
                ("scale", StandardScaler()),
                ("model", LogisticRegression(
                    penalty="l2",
                    solver="liblinear",
                    class_weight="balanced",
                    max_iter=5000,
                    random_state=random_state,
                )),
            ]),
            {"model__C": [0.01, 0.1, 1.0, 10.0]},
        ),
        "logistic_l1": (
            Pipeline([
                ("scale", StandardScaler()),
                ("model", LogisticRegression(
                    penalty="l1",
                    solver="liblinear",
                    class_weight="balanced",
                    max_iter=5000,
                    random_state=random_state,
                )),
            ]),
            {"model__C": [0.01, 0.1, 1.0, 10.0]},
        ),
        "svm_rbf": (
            Pipeline([
                ("scale", StandardScaler()),
                ("model", SVC(
                    kernel="rbf",
                    probability=True,
                    class_weight="balanced",
                    random_state=random_state,
                )),
            ]),
            {
                "model__C": [0.1, 1.0, 10.0],
                "model__gamma": ["scale", 0.01, 0.001],
            },
        ),
        "random_forest": (
            RandomForestClassifier(
                n_estimators=500,
                max_features="sqrt",
                class_weight="balanced",
                random_state=random_state,
                n_jobs=-1,
            ),
            {
                "max_depth": [None, 3, 5],
                "min_samples_leaf": [2, 3, 5],
            },
        ),
    }


def tune_model(model, param_grid, X_train, y_train):
    inner_cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
    search = GridSearchCV(
        model,
        param_grid=param_grid,
        scoring="roc_auc",
        cv=inner_cv,
        n_jobs=-1,
    )
    search.fit(X_train, y_train)
    return search.best_estimator_, search.best_params_


## Single-omics Evaluation

In [15]:
def evaluate_single_omics(name, df, prior_mode="pax"):
    X, y = split_x_class_y(df)
    cv = RepeatedStratifiedKFold(
        n_splits=N_SPLITS,
        n_repeats=N_REPEATS,
        random_state=RANDOM_STATE,
    )
    models = make_classification_models()

    rows = []
    predictions = []

    for fold_id, (train_idx, test_idx) in enumerate(cv.split(X, y), start=1):
        X_train_raw = X.iloc[train_idx]
        X_test_raw = X.iloc[test_idx]
        y_train = y.iloc[train_idx]
        y_test = y.iloc[test_idx]

        selected = select_features_by_omics(name, X_train_raw, prior_mode=prior_mode, pax_genes=pax_genes, nest_genes=nest_genes)
        X_train = X_train_raw[selected]
        X_test = X_test_raw[selected]

        for model_name, (model, grid) in models.items():
            best_model, best_params = tune_model(clone(model), grid, X_train, y_train)
            y_pred = best_model.predict(X_test)
            y_score = positive_class_score(best_model, X_test)
            metrics = classification_metrics(y_test, y_pred, y_score)

            rows.append({
                "analysis": "single_omics",
                "omics": name,
                "prior_mode": prior_mode,
                "model": model_name,
                "fold": fold_id,
                "n_train": len(train_idx),
                "n_test": len(test_idx),
                "n_features": len(selected),
                "n_pax_genes": len(set(selected) & pax_genes),
                "n_nest_genes": len(set(selected) & nest_genes),
                "best_params": str(best_params),
                **metrics,
            })

            predictions.extend([
                {
                    "analysis": "single_omics",
                    "omics": name,
                    "prior_mode": prior_mode,
                    "model": model_name,
                    "fold": fold_id,
                    "sample_id": sample_id,
                    "y_true": int(true),
                    "y_true_label": "Sensitive" if true == 1 else "Resistant",
                    "y_pred": int(pred),
                    "y_pred_label": "Sensitive" if pred == 1 else "Resistant",
                    "y_score_sensitive": score,
                }
                for sample_id, true, pred, score in zip(X_test.index, y_test, y_pred, y_score)
            ])

    return pd.DataFrame(rows), pd.DataFrame(predictions)


## Run Single-omics Baselines

This evaluates RNA on all available RNA samples, and RNA/CNV/mutation on the 62-sample paired cohort for fair comparison.

In [16]:
single_results = []
single_predictions = []

single_jobs = [
    ("rna_all", rna_df),
    ("rna_paired", rna_paired_df),
    ("cnv", cnv_paired_df),
    ("mutation", mutation_paired_df),
]

for name, df in single_jobs:
    print(f"Running {name}...")
    result_df, pred_df = evaluate_single_omics(name, df, prior_mode="pax")
    single_results.append(result_df)
    single_predictions.append(pred_df)

single_results_df = pd.concat(single_results, ignore_index=True)
single_predictions_df = pd.concat(single_predictions, ignore_index=True)

single_results_df.to_csv(RESULTS_DIR / "single_omics_classification_cv_results.csv", index=False)
single_predictions_df.to_csv(RESULTS_DIR / "single_omics_classification_cv_predictions.csv", index=False)

single_results_df.groupby(["omics", "model"])[[
    "auroc", "auprc", "balanced_accuracy", "f1", "sensitivity", "specificity", "n_features"
]].mean().round(3)


Running rna_all...
Running rna_paired...
Running cnv...
Running mutation...


auroc  auprc  balanced_accuracy     f1  sensitivity  \
omics      model                                                                
cnv        logistic_l1    0.448  0.475              0.465  0.247        0.273   
           logistic_l2    0.455  0.503              0.474  0.469        0.509   
           random_forest  0.534  0.590              0.527  0.514        0.556   
           svm_rbf        0.387  0.472              0.559  0.432        0.513   
mutation   logistic_l1    0.445  0.466              0.472  0.139        0.157   
           logistic_l2    0.389  0.477              0.436  0.447        0.532   
           random_forest  0.396  0.473              0.428  0.420        0.488   
           svm_rbf        0.500  0.565              0.487  0.321        0.496   
rna_all    logistic_l1    0.494  0.550              0.492  0.411        0.442   
           logistic_l2    0.637  0.691              0.575  0.558        0.578   
           random_forest  0.652  0.674              0.620  0.596        0.593   
           svm_rbf        0.428  0.531              0.539  0.521        0.603   
rna_paired logistic_l1    0.510  0.567              0.500  0.391        0.408   
           logistic_l2    0.643  0.688              0.613  0.611        0.663   
           random_forest  0.638  0.661              0.604  0.538        0.513   
           svm_rbf        0.505  0.582              0.571  0.493        0.533   

                          specificity  n_features  
omics      model                                   
cnv        logistic_l1          0.657      401.16  
           logistic_l2          0.439      401.16  
           random_forest        0.498      401.16  
           svm_rbf              0.604      401.16  
mutation   logistic_l1          0.786      491.92  
           logistic_l2          0.339      491.92  
           random_forest        0.368      491.92  
           svm_rbf              0.477      491.92  
rna_all    logistic_l1          0.543     2054.88  
           logistic_l2          0.571     2054.88  
           random_forest        0.646     2054.88  
           svm_rbf              0.474     2054.88  
rna_paired logistic_l1          0.592     2055.68  
           logistic_l2          0.564     2055.68  
           random_forest        0.695     2055.68  
           svm_rbf              0.609     2055.68

## Early-fusion Multi-omics Evaluation

For each fold, feature selection is performed independently within RNA, CNV, and mutation training folds. The selected matrices are concatenated and evaluated as a single feature table.

In [17]:
def evaluate_early_fusion(prior_mode="pax"):
    X_rna, y = split_x_class_y(rna_paired_df)
    X_cnv, _ = split_x_class_y(cnv_paired_df)
    X_mut, _ = split_x_class_y(mutation_paired_df)

    cv = RepeatedStratifiedKFold(
        n_splits=N_SPLITS,
        n_repeats=N_REPEATS,
        random_state=RANDOM_STATE,
    )
    models = make_classification_models()

    rows = []
    predictions = []

    for fold_id, (train_idx, test_idx) in enumerate(cv.split(X_rna, y), start=1):
        y_train = y.iloc[train_idx]
        y_test = y.iloc[test_idx]

        selected_rna = select_features_by_omics("rna_paired", X_rna.iloc[train_idx], prior_mode=prior_mode, pax_genes=pax_genes, nest_genes=nest_genes)
        selected_cnv = select_features_by_omics("cnv", X_cnv.iloc[train_idx], prior_mode=prior_mode, pax_genes=pax_genes, nest_genes=nest_genes)
        selected_mut = select_features_by_omics("mutation", X_mut.iloc[train_idx], prior_mode=prior_mode, pax_genes=pax_genes, nest_genes=nest_genes)

        X_train = pd.concat(
            [
                X_rna.iloc[train_idx][selected_rna].add_prefix("rna__"),
                X_cnv.iloc[train_idx][selected_cnv].add_prefix("cnv__"),
                X_mut.iloc[train_idx][selected_mut].add_prefix("mut__"),
            ],
            axis=1,
        )
        X_test = pd.concat(
            [
                X_rna.iloc[test_idx][selected_rna].add_prefix("rna__"),
                X_cnv.iloc[test_idx][selected_cnv].add_prefix("cnv__"),
                X_mut.iloc[test_idx][selected_mut].add_prefix("mut__"),
            ],
            axis=1,
        )

        for model_name, (model, grid) in models.items():
            best_model, best_params = tune_model(clone(model), grid, X_train, y_train)
            y_pred = best_model.predict(X_test)
            y_score = positive_class_score(best_model, X_test)
            metrics = classification_metrics(y_test, y_pred, y_score)

            rows.append({
                "analysis": "early_fusion",
                "omics": "rna_cnv_mutation",
                "prior_mode": prior_mode,
                "model": model_name,
                "fold": fold_id,
                "n_train": len(train_idx),
                "n_test": len(test_idx),
                "n_features": X_train.shape[1],
                "n_rna_features": len(selected_rna),
                "n_cnv_features": len(selected_cnv),
                "n_mutation_features": len(selected_mut),
                "best_params": str(best_params),
                **metrics,
            })

            predictions.extend([
                {
                    "analysis": "early_fusion",
                    "omics": "rna_cnv_mutation",
                    "prior_mode": prior_mode,
                    "model": model_name,
                    "fold": fold_id,
                    "sample_id": sample_id,
                    "y_true": int(true),
                    "y_true_label": "Sensitive" if true == 1 else "Resistant",
                    "y_pred": int(pred),
                    "y_pred_label": "Sensitive" if pred == 1 else "Resistant",
                    "y_score_sensitive": score,
                }
                for sample_id, true, pred, score in zip(X_test.index, y_test, y_pred, y_score)
            ])

    return pd.DataFrame(rows), pd.DataFrame(predictions)


In [18]:
print("Running early fusion...")
early_results_df, early_predictions_df = evaluate_early_fusion(prior_mode="pax")

early_results_df.to_csv(RESULTS_DIR / "early_fusion_classification_cv_results.csv", index=False)
early_predictions_df.to_csv(RESULTS_DIR / "early_fusion_classification_cv_predictions.csv", index=False)

early_results_df.groupby(["model"])[[
    "auroc", "auprc", "balanced_accuracy", "f1", "sensitivity", "specificity", "n_features"
]].mean().round(3)


Running early fusion...


,auroc,auprc,balanced_accuracy,f1,sensitivity,specificity,n_features
model,,,,,,,
logistic_l1,0.452,0.511,0.444,0.314,0.321,0.568,2948.76
logistic_l2,0.636,0.668,0.590,0.602,0.685,0.495,2948.76
random_forest,0.658,0.667,0.613,0.555,0.536,0.690,2948.76
svm_rbf,0.439,0.524,0.574,0.426,0.472,0.675,2948.76


## Late-fusion / Stacking Multi-omics Evaluation

Late fusion trains one base classifier per omics layer and combines their out-of-fold probability predictions with a simple meta-classifier. This tests whether CNV and mutation add complementary information without concatenating thousands of features directly with RNA.

For runtime, the default stack uses `logistic_l2` base models and a ridge-like logistic meta-model. The implementation keeps feature selection inside the relevant training folds.

In [19]:
def get_model_spec(model_name):
    models = make_classification_models()
    if model_name not in models:
        raise ValueError(f"Unknown model_name: {model_name}")
    return models[model_name]


def fit_predict_one_omics(omics_name, X_train_raw, y_train, X_test_raw, model_name, prior_mode):
    selected = select_features_by_omics(
        omics_name,
        X_train_raw,
        prior_mode=prior_mode,
        pax_genes=pax_genes,
        nest_genes=nest_genes,
    )
    model, grid = get_model_spec(model_name)
    best_model, best_params = tune_model(clone(model), grid, X_train_raw[selected], y_train)
    test_score = positive_class_score(best_model, X_test_raw[selected])
    return test_score, selected, best_params


def build_oof_base_predictions(X_layers, y_train, base_model_name="logistic_l2", prior_mode="pax"):
    inner_cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
    oof = pd.DataFrame(index=y_train.index)

    for omics_name, X_train_raw in X_layers.items():
        scores = pd.Series(index=y_train.index, dtype=float)
        for inner_train_idx, inner_valid_idx in inner_cv.split(X_train_raw, y_train):
            X_inner_train = X_train_raw.iloc[inner_train_idx]
            X_inner_valid = X_train_raw.iloc[inner_valid_idx]
            y_inner_train = y_train.iloc[inner_train_idx]

            selected = select_features_by_omics(
                omics_name,
                X_inner_train,
                prior_mode=prior_mode,
                pax_genes=pax_genes,
                nest_genes=nest_genes,
            )
            model, grid = get_model_spec(base_model_name)
            best_model, _ = tune_model(
                clone(model),
                grid,
                X_inner_train[selected],
                y_inner_train,
            )
            scores.iloc[inner_valid_idx] = positive_class_score(best_model, X_inner_valid[selected])

        oof[omics_name] = scores

    return oof


def evaluate_late_fusion(base_model_name="logistic_l2", prior_mode="pax"):
    X_rna, y = split_x_class_y(rna_paired_df)
    X_cnv, _ = split_x_class_y(cnv_paired_df)
    X_mut, _ = split_x_class_y(mutation_paired_df)

    cv = RepeatedStratifiedKFold(
        n_splits=N_SPLITS,
        n_repeats=N_REPEATS,
        random_state=RANDOM_STATE,
    )

    meta_model = Pipeline([
        ("scale", StandardScaler()),
        ("model", LogisticRegression(
            penalty="l2",
            solver="liblinear",
            class_weight="balanced",
            C=1.0,
            max_iter=5000,
            random_state=RANDOM_STATE,
        )),
    ])

    rows = []
    predictions = []

    for fold_id, (train_idx, test_idx) in enumerate(cv.split(X_rna, y), start=1):
        y_train = y.iloc[train_idx]
        y_test = y.iloc[test_idx]

        train_layers = {
            "rna_paired": X_rna.iloc[train_idx],
            "cnv": X_cnv.iloc[train_idx],
            "mutation": X_mut.iloc[train_idx],
        }
        test_layers = {
            "rna_paired": X_rna.iloc[test_idx],
            "cnv": X_cnv.iloc[test_idx],
            "mutation": X_mut.iloc[test_idx],
        }

        meta_train = build_oof_base_predictions(
            train_layers,
            y_train,
            base_model_name=base_model_name,
            prior_mode=prior_mode,
        )

        meta_test = pd.DataFrame(index=y_test.index)
        feature_counts = {}
        pax_counts = {}
        nest_counts = {}
        base_params = {}

        for omics_name in ["rna_paired", "cnv", "mutation"]:
            test_score, selected, best_params = fit_predict_one_omics(
                omics_name,
                train_layers[omics_name],
                y_train,
                test_layers[omics_name],
                model_name=base_model_name,
                prior_mode=prior_mode,
            )
            meta_test[omics_name] = test_score
            feature_counts[omics_name] = len(selected)
            pax_counts[omics_name] = len(set(selected) & pax_genes)
            nest_counts[omics_name] = len(set(selected) & nest_genes)
            base_params[omics_name] = best_params

        meta_model.fit(meta_train, y_train)
        y_score = positive_class_score(meta_model, meta_test)
        y_pred = meta_model.predict(meta_test)
        metrics = classification_metrics(y_test, y_pred, y_score)

        rows.append({
            "analysis": "late_fusion_stacking",
            "omics": "rna_cnv_mutation",
            "prior_mode": prior_mode,
            "model": f"stack_{base_model_name}",
            "fold": fold_id,
            "n_train": len(train_idx),
            "n_test": len(test_idx),
            "n_features": meta_train.shape[1],
            "n_rna_features": feature_counts["rna_paired"],
            "n_cnv_features": feature_counts["cnv"],
            "n_mutation_features": feature_counts["mutation"],
            "n_rna_pax_genes": pax_counts["rna_paired"],
            "n_cnv_pax_genes": pax_counts["cnv"],
            "n_mutation_pax_genes": pax_counts["mutation"],
            "base_model": base_model_name,
            "base_params": str(base_params),
            **metrics,
        })

        predictions.extend([
            {
                "analysis": "late_fusion_stacking",
                "omics": "rna_cnv_mutation",
                "prior_mode": prior_mode,
                "model": f"stack_{base_model_name}",
                "fold": fold_id,
                "sample_id": sample_id,
                "y_true": int(true),
                "y_true_label": "Sensitive" if true == 1 else "Resistant",
                "y_pred": int(pred),
                "y_pred_label": "Sensitive" if pred == 1 else "Resistant",
                "y_score_sensitive": score,
                "rna_score": meta_test.loc[sample_id, "rna_paired"],
                "cnv_score": meta_test.loc[sample_id, "cnv"],
                "mutation_score": meta_test.loc[sample_id, "mutation"],
            }
            for sample_id, true, pred, score in zip(y_test.index, y_test, y_pred, y_score)
        ])

    return pd.DataFrame(rows), pd.DataFrame(predictions)


In [20]:
print("Running late fusion / stacking...")
late_results_df, late_predictions_df = evaluate_late_fusion(
    base_model_name="logistic_l2",
    prior_mode="pax",
)

late_results_df.to_csv(RESULTS_DIR / "late_fusion_classification_cv_results.csv", index=False)
late_predictions_df.to_csv(RESULTS_DIR / "late_fusion_classification_cv_predictions.csv", index=False)

late_results_df.groupby(["model"])[[
    "auroc", "auprc", "balanced_accuracy", "f1", "sensitivity", "specificity", "n_features"
]].mean().round(3)


Running late fusion / stacking...


,auroc,auprc,balanced_accuracy,f1,sensitivity,specificity,n_features
model,,,,,,,
stack_logistic_l2,0.544,0.606,0.538,0.519,0.549,0.528,3.0


In [ ]:
RUN_STACKING_SENSITIVITY = False

if RUN_STACKING_SENSITIVITY:

    late_results = []
    late_predictions = []

    for base_model_name in ["logistic_l2", "random_forest", "svm_rbf"]:
        print(f"Running late fusion / stacking with {base_model_name}...")
        result_df, pred_df = evaluate_late_fusion(
            base_model_name=base_model_name,
            prior_mode="pax",
        )
        late_results.append(result_df)
        late_predictions.append(pred_df)

    late_results_df = pd.concat(late_results, ignore_index=True)
    late_predictions_df = pd.concat(late_predictions, ignore_index=True)

    late_results_df.to_csv(RESULTS_DIR / "late_fusion_classification_cv_results.csv", index=False)
    late_predictions_df.to_csv(RESULTS_DIR / "late_fusion_classification_cv_predictions.csv", index=False)

    late_results_df.groupby(["model"])[[
        "auroc", "auprc", "balanced_accuracy", "f1", "sensitivity", "specificity", "n_features"
    ]].mean().round(3)

Running late fusion / stacking with logistic_l2...
Running late fusion / stacking with random_forest...
Running late fusion / stacking with svm_rbf...


,auroc,auprc,balanced_accuracy,f1,sensitivity,specificity,n_features
model,,,,,,,
stack_logistic_l2,0.544,0.606,0.538,0.519,0.549,0.528,3.0
stack_random_forest,0.529,0.579,0.517,0.482,0.492,0.543,3.0
stack_svm_rbf,0.518,0.571,0.474,0.345,0.493,0.454,3.0


## Compare Model Families

In [23]:
all_results_df = pd.concat([single_results_df, early_results_df, late_results_df], ignore_index=True)
summary_df = (
    all_results_df
    .groupby(["analysis", "omics", "model"])
    .agg(
        auroc_mean=("auroc", "mean"),
        auroc_sd=("auroc", "std"),
        auprc_mean=("auprc", "mean"),
        auprc_sd=("auprc", "std"),
        balanced_accuracy_mean=("balanced_accuracy", "mean"),
        balanced_accuracy_sd=("balanced_accuracy", "std"),
        f1_mean=("f1", "mean"),
        sensitivity_mean=("sensitivity", "mean"),
        specificity_mean=("specificity", "mean"),
        n_features_mean=("n_features", "mean"),
    )
    .reset_index()
    .sort_values(["auroc_mean", "auprc_mean"], ascending=False)
)

summary_df.to_csv(RESULTS_DIR / "classification_model_summary.csv", index=False)
summary_df.round(3)


,analysis,omics,model,auroc_mean,auroc_sd,auprc_mean,auprc_sd,balanced_accuracy_mean,balanced_accuracy_sd,f1_mean,sensitivity_mean,specificity_mean,n_features_mean
2,early_fusion,rna_cnv_mutation,random_forest,0.658,0.134,0.667,0.112,0.613,0.121,0.555,0.536,0.690,2948.76
17,single_omics,rna_all,random_forest,0.652,0.134,0.674,0.117,0.620,0.127,0.596,0.593,0.646,2054.88
20,single_omics,rna_paired,logistic_l2,0.643,0.124,0.688,0.116,0.613,0.094,0.611,0.663,0.564,2055.68
21,single_omics,rna_paired,random_forest,0.638,0.128,0.661,0.109,0.604,0.102,0.538,0.513,0.695,2055.68
16,single_omics,rna_all,logistic_l2,0.637,0.143,0.691,0.120,0.575,0.122,0.558,0.578,0.571,2054.88
1,early_fusion,rna_cnv_mutation,logistic_l2,0.636,0.112,0.668,0.112,0.590,0.086,0.602,0.685,0.495,2948.76
4,late_fusion_stacking,rna_cnv_mutation,stack_logistic_l2,0.544,0.160,0.606,0.142,0.538,0.118,0.519,0.549,0.528,3.00
9,single_omics,cnv,random_forest,0.534,0.150,0.590,0.133,0.527,0.125,0.514,0.556,0.498,401.16
5,late_fusion_stacking,rna_cnv_mutation,stack_random_forest,0.529,0.152,0.579,0.119,0.517,0.125,0.482,0.492,0.543,3.00
6,late_fusion_stacking,rna_cnv_mutation,stack_svm_rbf,0.518,0.162,0.571,0.124,0.474,0.066,0.345,0.493,0.454,3.00


## Notes for Interpretation

Recommended first-pass readout:

1. Compare `rna_paired` vs `cnv` vs `mutation` to establish single-omics baselines.
2. Compare `early_fusion` against the best single-omics model.
3. If early fusion does not improve, use this as support for late-fusion/stacking as the next model family.
4. Use the saved fold-level predictions for paired statistical comparisons and ROC/PR plots.